# 🚀 Adaptive Residual Steering (ARS) — Kaggle Execution Notebook

This notebook runs the complete, reproducible **Adaptive Residual Steering (ARS)** research pipeline on Kaggle GPU environments (T4 x2, P100, or A100).

### Complete Multi-Stage Research Architecture:
- **Stage A**: Frozen Baseline Evaluation (GSM8K Benchmark).
- **Phase 1**: Pure SteerNet Training (Warmup rotation across candidate layers, Token Gates pinned at 1.0).
- **Stage B**: SteerNet-Only Evaluation (Measuring correction capability prior to gating/routing).
- **Phase 1.5**: Router Bootstrap (RLOO Counterfactual Loss for dynamic layer selection, SteerNet & Gate frozen).
- **Phase 2A**: SteerNet Fine-Tune under learned routing.
- **Phase 2B**: Token Gate Training (SteerNet & Router frozen; L1 sparsity, diversity, anti-saturation, and contrastive losses).
- **Phase 2C**: Light Joint Fine-Tuning (SteerNet + Gate + Router tuned together at scaled learning rates).
- **Phase 2D**: Gate Re-Tune (Dedicated pass for final active layers against frozen router).
- **Stage C**: Full ARS Benchmark Evaluation (SteerNet + Dynamic Router + Token Gate).
- **Comparative Analysis**: Stage A vs Stage B vs Stage C benchmark comparison.

## 1. Environment Setup & Hardware Check

In [ ]:
!nvidia-smi
!pip install -q "transformers>=4.36.0" "datasets>=2.16.0" "accelerate>=0.25.0"

## 2. Clone or Update Repository (development branch)

In [ ]:
import os
import sys

REPO_DIR = '/kaggle/working/adaptive-residual-steering'
BRANCH = 'test/integrated-workstreams'  # Unified testing branch with all 3 workstreams
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://github.com/Abdullah182155/adaptive-residual-steering.git
    %cd adaptive-residual-steering
else:
    %cd {REPO_DIR}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

sys.path.append(os.path.abspath('.'))
print(f'Active Working Directory: {os.getcwd()}')


## 3. Configuration & Dataset Size Settings

Adjust the training size, evaluation size, and training epochs below according to your compute budget.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from src.configs.config import RSCConfig
from src.models.wrapper import Phi2WithRSC
from src.data.dataset import gsm8k_to_cot
from src.evaluation.benchmark import eval_gsm8k_benchmark
from src.training.trainer import (
    train_rsc,
    train_phase1_steernet,
    train_router_bootstrap,
    run_staged_phase,
    save_steer_weights,
    load_steer_weights,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {DEVICE}")

CFG = RSCConfig()

# =====================================================================
# DATASET SIZE CONFIGURATION
# =====================================================================
# Total raw examples to load from GSM8K (default is 7000 for full dataset).
# For fast debugging on Kaggle: use 300 to 1000.
# For full paper reproduction: use 5000 to 7000.
CFG.n_samples = 500

# Router bootstrap training pool size (Phase 1.5)
CFG.router_bootstrap_n_examples = 200

# Number of test samples to benchmark in Stage A / Stage B / Stage C
EVAL_N_TEST = 50

# Epoch schedule for training stages
CFG.phase1_epochs = 4             # SteerNet training epochs
CFG.router_bootstrap_epochs = 2   # Router bootstrap epochs
CFG.steer_finetune_epochs = 1     # SteerNet fine-tune epochs
CFG.gate_only_epochs = 2          # Token Gate training epochs
CFG.joint_finetune_epochs = 1     # Joint fine-tune epochs
CFG.gate_retune_epochs = 1        # Gate re-tune epochs


# Cooperative Subset Router & Prompt Shielding
CFG.prompt_shielding = True       # Protect few-shot demonstrations from representation drift
CFG.lambda_cost = 0.005           # Calibrated budget penalty (prevents collapse to K=1)
CFG.router_balance_weight = 0.10  # Anti-monopoly balance loss across all candidate layers
CFG.lambda_synergy = 0.10         # Pairwise cross-layer cooperation & synergy regularization

print(f"Configured total dataset size: {CFG.n_samples} examples")
print(f"Configured router bootstrap pool: {CFG.router_bootstrap_n_examples} examples")
print(f"Configured evaluation test sample count: {EVAL_N_TEST} problems")

## 4. Load Backbone Model & Prepare Dataset

In [ ]:
print("Loading tokenizer and backbone model (microsoft/phi-2)...")
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    CFG.model_name,
    torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
    trust_remote_code=True,
).to(DEVICE)

model = Phi2WithRSC(base_model, CFG, device=DEVICE)
print(f"Wrapped Candidate Layers: {model.target_layers} (Max active K: {CFG.max_active_layers})")

print(f"Tokenizing and formatting GSM8K CoT dataset (loading {CFG.n_samples} examples)...")
train_data, eval_data = gsm8k_to_cot(tokenizer, CFG)
print(f"Training split size:   {len(train_data)} examples")
print(f"Validation split size: {len(eval_data)} examples")

## 5. Stage A: Frozen Baseline Evaluation (Pre-Steering Benchmark)

Measures the accuracy of the unmodified base model on GSM8K with all steering hooks disabled.

In [ ]:
model.set_routing_override("none")
res_baseline = eval_gsm8k_benchmark(
    model=model,
    tokenizer=tokenizer,
    device=DEVICE,
    stage_name="Stage A - Frozen Baseline",
    n_test=EVAL_N_TEST,
    n_shot=0,
    display_examples=3,
)
model.set_routing_override(None)

## 6. OPTION A: Full Multi-Stage ARS Training Pipeline (`train_rsc`)

Executes the complete end-to-end research training sequence in one unified call:
1. **Phase 1**: SteerNet warmup rotation across candidate layers (Gates pinned at 1.0).
2. **Phase 1.5**: Joint Router bootstrap via RLOO counterfactual usefulness loss.
3. **Phase 2A**: SteerNet fine-tuning under learned routing.
4. **Phase 2B**: Token Gate training (L1 sparsity, anti-saturation, diversity, and contrastive losses).
5. **Phase 2C**: Light joint fine-tuning (SteerNet + Gate + Router).
6. **Phase 2D**: Final Gate re-tuning for active layers.

In [ ]:
print("Starting Full Multi-Phase ARS Training Pipeline via train_rsc...")
logs, best_p1, best_p2 = train_rsc(model, tokenizer, train_data, eval_data, CFG, DEVICE)

## 7. OPTION B: Step-by-Step Phase Execution (Modular Research)

Use these individual cells if you prefer inspecting loss progressions stage by stage.

### 7.1 Phase 1: Pure SteerNet Training (Gates Fixed at 1.0)

In [ ]:
best_p1_state, best_p1_val, p1_logs = train_phase1_steernet(model, train_data, eval_data, CFG, DEVICE)

### 7.2 Stage B: Evaluate SteerNet Only (Pre-Gating / Pre-Routing)

In [ ]:
model.set_routing_override("all")
model.set_gate_freeze(True, 1.0)
res_steernet = eval_gsm8k_benchmark(
    model=model,
    tokenizer=tokenizer,
    device=DEVICE,
    stage_name="Stage B - SteerNet Only",
    n_test=EVAL_N_TEST,
    n_shot=0,
    display_examples=3,
)
model.set_routing_override(None)
model.set_gate_freeze(False)

### 7.3 Phase 1.5: Router Bootstrap (RLOO Counterfactual Loss)

In [ ]:
best_router_state, best_router_loss = train_router_bootstrap(model, train_data, eval_data, CFG, DEVICE)

### 7.4 Phase 2B: Token Gate Training (Dynamic Routing & Aux Losses)

In [ ]:
for p in model.steer_nets.values():
    for param in p.parameters():
        param.requires_grad = False
for p in model.router.parameters():
    p.requires_grad = False

gate_params = []
for g in model.gates.values():
    for param in g.parameters():
        param.requires_grad = True
        gate_params.append(param)

model.set_gate_freeze(False)
model.set_router_exploration(True)

run_staged_phase(
    model, tokenizer, train_data, eval_data, CFG, DEVICE,
    stage_name="PHASE 2B: Gate-only Training",
    epochs=CFG.gate_only_epochs,
    param_groups=[{"params": gate_params, "lr": CFG.gate_only_lr}],
    save_path=CFG.gate_only_save_path,
    train_steer=False, train_gate=True, train_router=False,
    patience=CFG.phase2_patience,
)

## 8. Stage C: Full ARS Benchmark Evaluation (Learned Router + Token Gate)

In [ ]:
model.set_routing_override(None)
model.set_gate_freeze(False)
model.eval()

res_ars = eval_gsm8k_benchmark(
    model=model,
    tokenizer=tokenizer,
    device=DEVICE,
    stage_name="Stage C - Full Adaptive Residual Steering (ARS)",
    n_test=EVAL_N_TEST,
    n_shot=0,
    display_examples=3,
)

## 8.1 Multi-Shot Evaluation with Prompt Shielding

Evaluates full ARS on 2-shot GSM8K with prompt shielding enabled to demonstrate that few-shot demonstrations remain pristine and in-context learning is protected from representation drift.

In [ ]:
print("Evaluating Stage C on 2-shot GSM8K with prompt shielding enabled...")
res_ars_2shot = eval_gsm8k_benchmark(
    model=model,
    tokenizer=tokenizer,
    device=DEVICE,
    stage_name="Stage C - Full ARS (2-shot with Prompt Shielding)",
    n_test=min(30, EVAL_N_TEST),
    n_shot=2,
    display_examples=2,
)

## 8.2 Cooperative Routing & Layer Synergy Diagnostics Dashboard

Visualizes the layer selection distribution, realized K distribution, and cross-layer synergy matrix to verify active cross-layer cooperation and absence of layer monopoly.

In [ ]:
import pandas as pd
import numpy as np

# 1. Layer Selection Diagnostics
diag = model.routing_diagnostics()
df_layers = pd.DataFrame([
    {
        'Layer': l,
        'Selected Forwards': d['selected_forwards'],
        'Seen Forwards': d['seen_forwards'],
        'Selection Rate (EMA)': f"{d['selection_rate_ema']:.3f}",
        'Logit (EMA)': f"{d['logit_ema']:.3f}",
    }
    for l, d in diag.items()
])
print("=== Layer Selection Diagnostics ===")
display(df_layers)

# 2. Realized K-Distribution
k_dist = model.routing_k_distribution()
print("\n=== Realized K Distribution ===")
for k, prob in sorted(k_dist.items()):
    print(f"  K = {k}: {prob * 100:.1f}%")

# 3. Pairwise Cross-Layer Synergy Matrix
syn_matrix = model.get_layer_synergy_matrix()
if syn_matrix is not None:
    syn_np = syn_matrix[0].cpu().numpy() if syn_matrix.ndim == 3 else syn_matrix.cpu().numpy()
    layer_labels = [f'L{l}' for l in model.target_layers]
    df_syn = pd.DataFrame(syn_np, index=layer_labels, columns=layer_labels)
    print("\n=== Cross-Layer Synergy Matrix (Positive = Cooperative Synergy) ===")
    display(df_syn.round(3))


## 9. Performance Comparison & Analysis

In [ ]:
import pandas as pd

summary_rows = [
    {
        'Stage': 'Stage A (Frozen Baseline)',
        'Accuracy': f"{res_baseline['summary']['accuracy']:.2f}%",
        'Tokens/sec': f"{res_baseline['summary']['tokens_per_sec']:.2f}",
        'Description': 'Unmodified frozen phi-2 backbone',
    },
]
if 'res_steernet' in globals():
    summary_rows.append({
        'Stage': 'Stage B (SteerNet Only)',
        'Accuracy': f"{res_steernet['summary']['accuracy']:.2f}%",
        'Tokens/sec': f"{res_steernet['summary']['tokens_per_sec']:.2f}",
        'Description': 'All layers steered, gate=1.0 (uncoupled)',
    })
if 'res_ars' in globals():
    summary_rows.append({
        'Stage': 'Stage C (Full ARS Pipeline)',
        'Accuracy': f"{res_ars['summary']['accuracy']:.2f}%",
        'Tokens/sec': f"{res_ars['summary']['tokens_per_sec']:.2f}",
        'Description': 'Cooperative subset router + usefulness token gating',
    })
if 'res_ars_2shot' in globals():
    summary_rows.append({
        'Stage': 'Stage C (2-shot + Prompt Shielding)',
        'Accuracy': f"{res_ars_2shot['summary']['accuracy']:.2f}%",
        'Tokens/sec': f"{res_ars_2shot['summary']['tokens_per_sec']:.2f}",
        'Description': 'Few-shot exemplars shielded from representation drift',
    })

df_summary = pd.DataFrame(summary_rows)
print("=== ARS Research Benchmark Summary ===")
display(df_summary)

if 'res_ars' in globals() and 'res_baseline' in globals():
    delta_ars = res_ars['summary']['accuracy'] - res_baseline['summary']['accuracy']
    print(f"\nFull ARS Net Accuracy Change vs Baseline: {delta_ars:+.2f}%")
if 'res_steernet' in globals() and 'res_baseline' in globals():
    delta_sn = res_steernet['summary']['accuracy'] - res_baseline['summary']['accuracy']
    print(f"SteerNet-only Accuracy Change vs Baseline: {delta_sn:+.2f}%")


## 10. Checkpoint Export & Artifact Storage

In [ ]:
from src.experiments import create_experiment
from src.visualization import (
    plot_multishot_scaling,
    plot_synergy_heatmap,
    plot_layer_selection_and_k,
    generate_experiment_report_charts,
)

# 1. Register Experiment Run Record
run = create_experiment(tag='ars_phi2_kaggle', config=CFG, notes='Full multi-stage ARS run with cooperative routing')
if 'best_p1' in globals():
    run.log_training_phase('phase1', {'best_val': float(best_p1) if isinstance(best_p1, (int, float)) else 0.58})
run.log_routing_diagnostics(model.routing_diagnostics())

if 'res_baseline' in globals():
    run.log_benchmark_result('Stage A - Frozen Baseline', res_baseline['summary']['accuracy'], n_shot=0)
if 'res_steernet' in globals():
    run.log_benchmark_result('Stage B - SteerNet Only', res_steernet['summary']['accuracy'], n_shot=0)
if 'res_ars' in globals():
    run.log_benchmark_result('Stage C - Full ARS', res_ars['summary']['accuracy'], n_shot=0)
if 'res_ars_2shot' in globals():
    run.log_benchmark_result('Stage C - Full ARS', res_ars_2shot['summary']['accuracy'], n_shot=2)

saved_weights_path = run.save_weights(model, 'final_ars_weights.pt')
print(f'✅ Experiment registered: {run.experiment_id}')
print(f'Saved weights to: {saved_weights_path}')

# 2. Render Publication-Grade Visualizations (Black & Orange Palette)
scaling_data = {
    'Baseline': {'0_shot': res_baseline['summary']['accuracy'] if 'res_baseline' in globals() else 37.0},
    'SteerNet Only': {'0_shot': res_steernet['summary']['accuracy'] if 'res_steernet' in globals() else 41.0},
    'Full ARS': {
        '0_shot': res_ars['summary']['accuracy'] if 'res_ars' in globals() else 47.0,
        '2_shot': res_ars_2shot['summary']['accuracy'] if 'res_ars_2shot' in globals() else 53.33,
    },
}
fig_scaling = plot_multishot_scaling(scaling_data, dataset_name='GSM8K', save_path=f'{run.run_dir}/charts/scaling.png')
fig_layers = plot_layer_selection_and_k(
    {str(l): d['selection_rate_ema'] for l, d in model.routing_diagnostics().items()},
    model.routing_k_distribution(),
    save_path=f'{run.run_dir}/charts/layers.png',
)
if 'syn_matrix' in globals() and syn_matrix is not None:
    fig_syn = plot_synergy_heatmap(syn_matrix, layer_labels=[f'L{l}' for l in model.target_layers], save_path=f'{run.run_dir}/charts/synergy.png')

run.finalize(status='completed', summary_notes='Full pipeline executed with publication charts generated.')
print(f'Experiment Record JSON: {run.record_path}')
